In [190]:
import os

from azure.cognitiveservices.vision.computervision import ComputerVisionClient
from azure.cognitiveservices.vision.computervision.models import OperationStatusCodes
from azure.cognitiveservices.vision.computervision.models import VisualFeatureTypes
from msrest.authentication import CognitiveServicesCredentials
import time
import cv2
import json
import time
import shapely
import sys
import numpy as np

# Authenticate

In [191]:
'''
Authenticate
Authenticates your credentials and creates a client.
'''
credidential = json.load(open("credidentiale.json"))
subscription_key = credidential["API_KEY"]
endpoint = credidential["END_POINT"]
computervision_client = ComputerVisionClient(endpoint, CognitiveServicesCredentials(subscription_key))
'''
END - Authenticate
'''

'\nEND - Authenticate\n'

# Problema 1
## calitatea procesului de recunoastere a textului, atat la nivel de caracter, cat si la nivel de cuvant:
- prin folosirea unei metrici de distanta sau
- prin folosirea mai multor metrici de distanta.

# Distance functions

In [192]:
def hammming_distance(ref_text, reg_text):
    i=0
    count=0
    while(i<min(len(ref_text),len(reg_text))):
        if ref_text[i]!=reg_text[i]:
            count+=1
        i+=1
    
    if i < len(reg_text):
        count += len(reg_text) - i
        
    return count

def jaro_winkler_distance(ref_text, reg_text):
    st1 = ref_text.replace("\n", "")
    st2 = reg_text.replace("\n", "")
    
    #aceasta este constanta din formula  Jaro-Winkler = jaro + L * p * (1 - jaro) ; L este lungimea prefixului comun ; \
    # formula pentru similaritatea Jaro-Winkler 
    p = 0.1
    
    #calul jaro:
    if len(st1) < len(st2):
        st1, st2 = st2, st1
    len1, len2 = len(st1), len(st2)
    
    if len2 == 0:
        return 0.0
    
    delta = max(len1, len2) // 2 - 1
    flag = [False for _ in range(len(st2))] #flag pentru a verifica daca un caracter a fost deja folosit
    
    ch1_match = []
    for i, ch1 in enumerate(st1):
        for j, ch2 in enumerate(st2):
            if j <= i + delta and j >= i - delta and ch1==ch2 and not flag[j]:
                flag[j] = True
                ch1_match.append(ch1)
                break
                
    matches = len(ch1_match)
    if matches == 0:
        return 1.0
    
    transpositions, id = 0,0
    for i, ch2 in enumerate(st2):
        if flag[i]:
            transpositions += (ch2 != ch1_match[id])
            id += 1
            
    jaro = (matches / len1 + matches / len2 + (matches - transpositions) / matches) / 3
    
    #calcul jaro-winkler
    prefix = 0
    for i in range(min(4,len2)):
        prefix+=(st1[i]==st2[i])
    
    return 1.0 - (jaro + prefix * p * (1 - jaro))

def levenshtein_distance(ref_text, reg_text):
    m = len(ref_text)
    n = len(reg_text)

    dp = [[0 for _ in range(n + 1)] for _ in range(m + 1)]
 
    for i in range(m + 1):
        dp[i][0] = i
    for j in range(n + 1):
        dp[0][j] = j

    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if ref_text[i - 1] == reg_text[j - 1]:
                dp[i][j] = dp[i - 1][j - 1]
            else:
                dp[i][j] = 1 + min(dp[i][j - 1], dp[i - 1][j], dp[i - 1][j - 1])
                
    return dp[m][n]

def longest_common_subsequence(ref_text, reg_text):
    ref_chars = ref_text.replace("\n", "")
    reg_chars = reg_text.replace("\n", "")
    
    m = len(ref_text)
    n = len(reg_text)

    dp = [[0] * (n + 1) for x in range(m + 1)]

    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if ref_text[i - 1] == reg_text[j - 1]:
                dp[i][j] = dp[i - 1][j - 1] + 1
            else:
                dp[i][j] = max(dp[i - 1][j],
                               dp[i][j - 1])

    return dp[m][n]

def dice_sorensen_distance(ref_text, reg_text):
    ref_chars = ref_text.replace("\n", "")
    reg_chars = reg_text.replace("\n", "")
    
    #voi folosig bigrams 
    ref_bigrams = [ref_chars[i:i+2] for i in range(len(ref_chars)-1)]
    reg_bigrams = [reg_chars[i:i+2] for i in range(len(reg_chars)-1)]
    
    common_bigrams = 2 * len(set(ref_bigrams) & set(reg_bigrams))
    
    return 1 - common_bigrams / (len(ref_bigrams) + len(reg_bigrams))

def overlap_coefficient_distance(ref_text, reg_text):
    ref_chars = ref_text.replace("\n", "")
    reg_chars = reg_text.replace("\n", "")
    
    ref_chars = set(ref_chars)
    reg_chars = set(reg_chars)
    
    common_chars = len(ref_chars & reg_chars)
    
    return 1 - common_chars / min(len(ref_chars), len(reg_chars))

# Read Image function

In [193]:
def read_image(img):
    read_response = computervision_client.read_in_stream(
        image=img,
        mode="Handwritten",
        raw=True
    )
    # print(read_response.as_dict())
    
    operation_id = read_response.headers['Operation-Location'].split('/')[-1]
    while True:
        read_result = computervision_client.get_read_result(operation_id)
        if read_result.status not in ['notStarted', 'running']:
            break
        time.sleep(1)
    
    # Print the detected text, line by line
    text = ""
    if read_result.status == OperationStatusCodes.succeeded:
        for text_result in read_result.analyze_result.read_results:
            for line in text_result.lines:
                text += line.text + "\n"
    
    text = text[:-1]
    return text

# Image text recognition

In [194]:
recognized_text = read_image(open("preImage/pretest.jpg", "rb"))
print(f"Regoznized Text:\n{recognized_text}")
#1
# reference_text = "Google Cloud\nPlatform"
#2#
# reference_text = "Succes in rezolvarea\ntEMELOR la\nLABORAtoaree de\nInteligenta Artificiala!"
#3
# reference_text = "Iustin merge\nla magazin sa\nisi cumpere de\nmancare cu\nbanii pe care\nii ia dat\nmama de astazi"
#4
#reference_text = "Mesajul ăsta\neste scris\ncu\ncarioca și cu\nPixul"
#5
#reference_text = "OLX\nVÂND\nGOLD 4"
#6
#reference_text = "Natsuki-CHAN\nDă-te bă io~"
#pre
reference_text = "Acesta este un chenar !\nBogdan are mere.\nMERELE\nSUNT GALBENE.\nPERELE sunt verzi !"

Regoznized Text:
Acesta este un chenar !
ALL AE
Bogdan are mere.
MERE LE
SUNT GALBENE.
PERELE sunt verzi !


# Distante metrice
## Algoritmi de distanta: 
### - Hamming : spune numarul de caractere care difera intre doua sirurile de caractere pe acelasi index
### - Jaro-Winkler : Jaro verifica similaritatea dintre doua siruri , daca 2 care sunt la fel se alfa la max de un numar delta de caractere distanta unul de celalalt, Winkler adauga un bonus pentru prefixele comune pentru o similaritate mai mare
### - Levenshtein, : distanta minima de operatii pentru a transforma un sir in altul
### - Longest Common Subsequence : lungimea celui mai lung subsir comun
### - Dice-Sorensen : verifica similaritatea sirurilor prin cate biagrame comune au
### - Overlap Coefficient : verifica similaritatea sirurilor prin cate caractere comune au
### - 

In [195]:
def cer_wer():
    print(f"Hamming Distance: {hammming_distance(reference_text, recognized_text)}")
    
    print(f"Jaro-Winkler Distance: {jaro_winkler_distance(reference_text, recognized_text)}")
    
    print(f"Levenshtein Distance chars: {levenshtein_distance(reference_text.replace("\n",""), recognized_text.replace("\n",""))}")
    
    var = reference_text.replace("\n"," ").split(" ")
    word_list  = recognized_text.replace("\n"," ").split(" ")
    print(f"Levenshtein Distance words: : {levenshtein_distance(var,word_list)}")
    
    print(f"Longest Common Subsequence: {longest_common_subsequence(reference_text, recognized_text)}")
    
    print(f"Dice-Sorensen Distance: {dice_sorensen_distance(reference_text, recognized_text)}")
    
    print(f"Overlap Coefficient Distance: {overlap_coefficient_distance(reference_text, recognized_text)}")

cer_wer()

Hamming Distance: 60
Jaro-Winkler Distance: 0.14653679653679652
Levenshtein Distance chars: 7
Levenshtein Distance words: : 4
Longest Common Subsequence: 81
Dice-Sorensen Distance: 0.22012578616352196
Overlap Coefficient Distance: 0.0


# Problema 2 calitatea localizarii corecte a textului in imagine

# Drawing box function

In [196]:
def draw_ocr_results(image, text, pts, color=(0, 0, 255)):
	# unpack the points list
	topLeft = pts[0]
	topRight = pts[1]
	bottomRight = pts[2]
	bottomLeft = pts[3]
	# draw the bounding box of the detected text
	cv2.line(image, topLeft, topRight, color, 2)
	cv2.line(image, topRight, bottomRight, color, 2)
	cv2.line(image, bottomRight, bottomLeft, color, 2)
	cv2.line(image, bottomLeft, topLeft, color, 2)
	# draw the text itself

	cv2.putText(image, text, (topLeft[0], topLeft[1] - 10),
		cv2.FONT_HERSHEY_SIMPLEX, 2.0, (255,255,255), 4)
	# return the output image
	return image

def drawing_box(path):
    
    img = open(path,'rb')
        
    read_response = computervision_client.read_in_stream(
        image=img,
        mode="Handwritten",
        raw=True
    )
    # print(read_response.as_dict())
    
    operation_id = read_response.headers['Operation-Location'].split('/')[-1]
    while True:
        read_result = computervision_client.get_read_result(operation_id)
        if read_result.status not in ['notStarted', 'running']:
            break
        time.sleep(1)
    
    # Print the detected text, line by line
    image= cv2.imread(path)
    final = image.copy()

    coordonate = []
    if read_result.status == OperationStatusCodes.succeeded:
        for text_result in read_result.analyze_result.read_results:
            for line in text_result.lines:
                text = line.text
             
                box = list(map(int, line.bounding_box))
                
                (tlX, tlY, trX, trY, brX, brY, blX, blY) = box
                pts = ((tlX, tlY), (trX, trY), (brX, brY), (blX, blY))
                coordonate.append(pts)
                
                final = draw_ocr_results(final, text, pts)
    cv2.imwrite("./preImage/output.png",final)
    if(os.path.exists(img.name + "_output.png")):
        os.remove(img.name + "_output.png")
    os.rename("./preImage/output.png", img.name + "_output.png")
    return coordonate

# Punctele poligon AI

In [197]:
puncte_AI = drawing_box('preImage/pretest.jpg')
puncte_AI.remove(puncte_AI[1]) #remove ALL E , anomaly
print(puncte_AI)

[((1612, 1033), (2404, 2567), (2239, 2651), (1462, 1107)), ((1167, 1220), (1713, 2301), (1576, 2368), (1038, 1291)), ((682, 1478), (918, 1958), (794, 2016), (561, 1534)), ((1031, 2193), (1505, 3193), (1397, 3245), (918, 2244)), ((333, 1617), (891, 2724), (757, 2790), (208, 1679))]


# Punctele din poligon dorite

In [198]:
#Json puncte
with open('preImage/puncte.json', 'r') as file:
    data = json.load(file)

all_points_x = []
all_points_y = []

for key, value in data['_via_img_metadata'].items():
    for region in value['regions']:
        all_points_x.append(region['shape_attributes']['all_points_x'])
        all_points_y.append(region['shape_attributes']['all_points_y'])

puncte_Dorite = []
for x,y in zip(all_points_x, all_points_y):
    pol = (tuple(zip(x,y)))
    puncte_Dorite.append(pol)

print(puncte_Dorite)


[((1404, 1108), (1609, 1014), (2463, 2537), (2229, 2647)), ((1034, 1289), (1203, 1211), (1740, 2307), (1588, 2381)), ((685, 1461), (936, 1954), (776, 2036), (529, 1539)), ((907, 2266), (1059, 2192), (1523, 3165), (1375, 3251)), ((328, 1605), (924, 2713), (739, 2803), (168, 1708))]


# Calcul de Jaccard index (areaOvelap / area of Union) precizia localizarii textului

In [199]:
def JaccardIndex():
    procente = []
    for i in range(len(puncte_AI)):
        A = shapely.Polygon(puncte_AI[i])
        B = shapely.Polygon(puncte_Dorite[i])
        I = A.intersection(B)
        U = A.union(B)
        procent = I.area / U.area
        print(f"Jaccard index: {procent} for box {i}")
        procente.append(procent)
    
    return f"Media procentelor este: {np.mean(procente)}"

In [200]:
print(JaccardIndex())

Jaccard index: 0.7180482092407262 for box 0
Jaccard index: 0.8150315503888506 for box 1
Jaccard index: 0.7366656476103128 for box 2
Jaccard index: 0.7068479510870986 for box 3
Jaccard index: 0.7266309849246444 for box 4
Media procentelor este: 0.7406448686503266


# Problema 3 posibilitati de imbunatatire a recunoasterii textului

# Exista diferite metode pentru preprocesarea imaginiilor ca recunoasterea textului sa fie mai usoara: 0. Rotirea documentelor , alinierea , cropping  etc
    
## 1. Reverse Colors: Inversarea culorilor imaginii pentru a face textul mai vizibil

In [201]:
image = cv2.imread("preImage/pretest.jpg")
reverse = cv2.bitwise_not(image)

cv2.imwrite("preImage/Reverse.jpg", reverse)

True

## 2. Binarizare ( Gray image )

In [202]:
image = cv2.imread("preImage/pretest.jpg")
gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
thresh, gray = cv2.threshold(gray, 120, 200, cv2.THRESH_BINARY)

cv2.imwrite("preImage/Gray.jpg", gray)

True

## 3. Noise Reduction

In [203]:
image = cv2.imread("preImage/pretest.jpg")
kernel = np.ones((1,1),np.uint8)
noise = cv2.dilate(image, kernel, iterations = 1)
kernel = np.ones((1,1),np.uint8)
noise = cv2.erode(noise, kernel, iterations = 1)
noise = cv2.morphologyEx(noise, cv2.MORPH_CLOSE, kernel)
noise = cv2.medianBlur(noise,3)

cv2.imwrite("preImage/Noise.jpg", noise)

True

## 4. Dilatare si Eradare 

In [204]:
dilated = cv2.imread("preImage/pretest.jpg")
dilated = cv2.bitwise_not(dilated)
kernel = np.ones((2,2),np.uint8)
dilated = cv2.erode(dilated, kernel, iterations = 1)
dilated = cv2.bitwise_not(dilated)

cv2.imwrite("preImage/Dilated.jpg", dilated)

True

# Compartii :

In [205]:
dict = {}
dict.update({"Original":JaccardIndex()})

Jaccard index: 0.7180482092407262 for box 0
Jaccard index: 0.8150315503888506 for box 1
Jaccard index: 0.7366656476103128 for box 2
Jaccard index: 0.7068479510870986 for box 3
Jaccard index: 0.7266309849246444 for box 4


In [206]:
#reverse
puncte_AI = drawing_box('preImage/Reverse.jpg')
puncte_AI.remove(puncte_AI[0]) #remove GRAPHICS - deja ai-ul vede mai multe cuvinte 
puncte_AI.remove(puncte_AI[1]) #remove ALL E 
dict.update({"Reverse":JaccardIndex()})

Jaccard index: 0.7436022909253552 for box 0
Jaccard index: 0.8101789771028222 for box 1
Jaccard index: 0.742779985733211 for box 2
Jaccard index: 0.7487688762520397 for box 3
Jaccard index: 0.7189342393191566 for box 4


In [207]:
#gray
puncte_AI = drawing_box('preImage/Gray.jpg')

dict.update({"Gray":JaccardIndex()})

Jaccard index: 0.7207216781179054 for box 0
Jaccard index: 0.8135415314434159 for box 1
Jaccard index: 0.7176704705382071 for box 2
Jaccard index: 0.774695148289386 for box 3
Jaccard index: 0.70823107668386 for box 4


In [208]:
#noice
puncte_AI = drawing_box('preImage/Noise.jpg')
puncte_AI.remove(puncte_AI[0]) 
puncte_AI.remove(puncte_AI[1]) #remove ALL E , anomaly
dict.update({"Noise":JaccardIndex()})

Jaccard index: 0.7086988436970918 for box 0
Jaccard index: 0.8131110860873569 for box 1
Jaccard index: 0.7461021094466525 for box 2
Jaccard index: 0.7012994575130409 for box 3
Jaccard index: 0.7031319460937453 for box 4


In [209]:
#dilated
puncte_AI = drawing_box('preImage/Dilated.jpg')
puncte_AI.remove(puncte_AI[0])
puncte_AI.remove(puncte_AI[1]) #remove ALL E , anomaly
dict.update({"Dilated":JaccardIndex()})

Jaccard index: 0.7003933065275962 for box 0
Jaccard index: 0.8171120811576327 for box 1
Jaccard index: 0.7349892999082849 for box 2
Jaccard index: 0.723575870801627 for box 3
Jaccard index: 0.7080727438072562 for box 4


In [210]:
for key, value in dict.items():
    print(f"{key} : {value}")
    
print(f"Best: {max(dict, key=dict.get)}")
print(f"Worst: {min(dict, key=dict.get)}")

Original : Media procentelor este: 0.7406448686503266
Reverse : Media procentelor este: 0.752852873866517
Gray : Media procentelor este: 0.7469719810145549
Noise : Media procentelor este: 0.7344686885675774
Dilated : Media procentelor este: 0.7368286604404795
Best: Reverse
Worst: Noise
